# script 4: cargue a clickhaus y validacion del numero magico (minio-to-clickhouse)
en este ultimo script leemos la tabla de iceberg que tenemos en el minion y la mandamos a clickhaus usando dlt. al final hacemos la consulta de conteo para validar que nos de exactamente las 3,475,226 filas que pide el profe.

> **rubrica de 5.0:** `SELECT count(*) FROM default.taxis_iceberg` tiene que retornar 3475226 sin perder datos.

In [ ]:
import dlt
from pyiceberg.catalog import load_catalog
import clickhouse_connect

# 1. nos conectamos a messi para leer los metadatos de la tabla iceberg
catalog = load_catalog(
    "nessie",
    **{
        "uri": "http://nessie:19120/iceberg/main/",
        "s3.endpoint": "http://minio:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
        "s3.path-style-access": "true",
    }
)

print("conectando con el catalogo de messi y cargando tabla demo.taxis_iceberg...")
tabla_iceberg = catalog.load_table("demo.taxis_iceberg")
print(f"tabla cargada : {tabla_iceberg.name()}")
print(f"total de columnas en el esquema: {len(tabla_iceberg.schema().columns)}")

In [ ]:
# 2. recurso dlt para pasar los datos a clickhaus por lotes (streaming)
# leemos directamente por lotes con to_arrow_batch_reader para máxima velocidad y eficiencia de memoria
@dlt.resource(name="taxis_iceberg", write_disposition="replace")
def stream_iceberg():
    scan = tabla_iceberg.scan()
    for batch in scan.to_arrow_batch_reader():
        yield batch

# 3. pipeline de dlt con destino clickhaus
# las credenciales las saca automaticamente de .dlt/secrets.toml
pipeline_ch = dlt.pipeline(
    pipeline_name="iceberg_to_clickhouse",
    destination="clickhouse",
    dataset_name="default",
)

print("insertando los datos en clickhaus en streaming por lotes...")
info_ch = pipeline_ch.run(stream_iceberg)
print("=== reporte de carga a clickhaus ===")
print(info_ch)

In [ ]:
# 4. validacion del numero magico del profe (3,475,226 filas)
client = clickhouse_connect.get_client(host="clickhouse", port=8123, username="default", password="default")

tablas = client.query("SHOW TABLES FROM default LIKE 'default___taxis_iceberg'")
if tablas.result_rows:
    client.command("RENAME TABLE `default___taxis_iceberg` TO `taxis_iceberg`")

# hacemos el conteo con sql en clickhaus
res_count = client.query("SELECT count(*) FROM default.taxis_iceberg")
total_filas = res_count.result_rows[0][0]

print("=======================================================")
print(f"  total de registros en clickhaus: {total_filas:,}")
print("=======================================================")

assert total_filas == 3475226, f"ojo: esperabamos 3,475,226 y dio {total_filas}"
print("salio melo: la cantidad coincide exactamente con las 3,475,226 filas")

In [ ]:
# 5. mostramos las tablas y una muestra de los datos
print("tablas creadas en clickhaus:")
tablas = client.query("SHOW TABLES FROM default")
for t in tablas.result_rows:
    print(" -", t[0])

print("\nprimeras 5 filas para ver que todo quedo bien:")
muestra = client.query("SELECT * FROM default.taxis_iceberg LIMIT 5")
for col in muestra.column_names:
    print(col, end=" | ")
print("\n" + "-" * 80)